In [1]:
!pip install scikit-learn tqdm

In [ ]:
from google.colab import drive
import os
drive_dataset_path="/content/drive/MyDrive/balanced_dementia_datasets"
local_path="/content/balanced_dementia_datasets"
!cp -r "$drive_dataset_path" "$local_path"

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets,transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix,classification_report
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
class AddGaussianNoise(object):
    def __init__(self,mean=0.,std=1.):
        self.std=std
        self.mean=mean
    def __call__(self,tensor):
        return tensor+torch.randn(tensor.size())*self.std+self.mean
transform=transforms.Compose([transforms.Resize((128,128)),transforms.Grayscale(num_output_channels=1),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(mean=[0.5],std=[0.5]),AddGaussianNoise(0.,0.05)])
train_data=datasets.ImageFolder("/content/balanced_dementia_datasets/train",transform=transform)
val_data=datasets.ImageFolder("/content/balanced_dementia_datasets/val",transform=transform)
train_loader=DataLoader(train_data,batch_size=128,shuffle=True)
val_loader=DataLoader(val_data,batch_size=128,shuffle=False)
classes=train_data.classes

In [ ]:
import torch
import torch.nn as nn
import math
class CNNTransformer(nn.Module):
    def __init__(self,num_classes=4,dropout=0.1):
        super(CNNTransformer,self).__init__()
        self.cnn=nn.Sequential(nn.Conv2d(1,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                               nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                               nn.Dropout(dropout),
                               nn.Conv2d(64,128,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                               nn.Conv2d(128,256,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),
                               nn.Dropout(dropout))
        self.d_model=256
        self.num_patches=8*8
        self.pos_embedding=nn.Parameter(torch.randn(1,self.num_patches+1,self.d_model))
        self.cls_token=nn.Parameter(torch.randn(1,1,self.d_model))
        encoder_layer=nn.TransformerEncoderLayer(d_model=self.d_model,nhead=8,dim_feedforward=512,dropout=dropout,batch_first=True)
        self.transformer=nn.TransformerEncoder(encoder_layer,num_layers=4)
        self.classifier=nn.Sequential(nn.Linear(self.d_model,256),nn.ReLU(),nn.Dropout(0.3),nn.Linear(self.d_model,num_classes))
    def forward(self,x):
        b=x.shape[0]
        x=self.cnn(x)
        x=x.flatten(2).transpose(1,2)
        cls_tokens=self.cls_token.expand(b,-1,-1)
        x=torch.cat((cls_tokens,x),dim=1)
        x+=self.pos_embedding
        x=self.transformer(x)
        x=x[:,0]
        x=self.classifier(x)
        return x

In [ ]:
device=torch.device("cuda"if torch.cuda.is_available() else"cpu")
model=CNNTransformer().to(device)
criterion=nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer=optim.Adam(model.parameters(),lr=0.0003,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=50)
scaler=torch.amp.GradScaler(device='cuda')

In [ ]:
from tqdm import tqdm
import torch
epochs=50
train_losses=[]
val_losses=[]
train_accs=[]
val_accs=[]
for epoch in range(epochs):
    model.train()
    running_loss=0
    correct=0
    total=0
    loop=tqdm(train_loader,desc=f"Epoch {epoch+1}/{epochs}")
    for imgs,labels in loop:
        imgs=imgs.to(device)
        labels=labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            outputs=model(imgs)
            loss=criterion(outputs,labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        running_loss+=loss.item()
        _,preds=torch.max(outputs,1)
        correct+=(preds==labels).sum().item()
        total+=labels.size(0)
        loop.set_postfix(loss=loss.item())
    avg_loss=running_loss/len(train_loader)
    train_acc=correct/total
    train_losses.append(avg_loss)
    train_accs.append(train_acc)
    model.eval()
    running_loss=0
    correct=0
    total=0
    with torch.no_grad():
        vloop=tqdm(val_loader,desc="Validation",leave=False)
        for imgs,labels in vloop:
            imgs=imgs.to(device)
            labels=labels.to(device)
            outputs=model(imgs)
            loss=criterion(outputs,labels)
            running_loss+=loss.item()
            _,preds=torch.max(outputs,1)
            correct+=(preds==labels).sum().item()
            total+=labels.size(0)
    val_loss=running_loss/len(val_loader)
    val_acc=correct/total
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    scheduler.step()
    curr_lr=scheduler.get_last_lr()[0]
    print(f"Epoch {epoch+1}/{epochs} Average Loss: {avg_loss:.4f}|LR: {curr_lr:.6f}")

In [ ]:
model.eval()
y_true=[]
y_pred=[]
with torch.no_grad():
    for imgs,labels in val_loader:
        imgs=imgs.to(device)
        outputs=model(imgs)
        _,predicted=torch.max(outputs,1)
        y_true.extend(labels.numpy())
        y_pred.extend(predicted.cpu().numpy())

In [ ]:
cm=confusion_matrix(y_true,y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm,annot=True,fmt="d",xticklabels=classes,yticklabels=classes,cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()
print(classification_report(y_true,y_pred,target_names=classes))

In [ ]:
import matplotlib.pyplot as plt
epochs_range=range(1,epochs+1)
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(epochs_range,train_losses,label="Train Loss")
plt.plot(epochs_range,val_losses,label="Validation Loss")
plt.title("Loss Comparison")
plt.legend()
plt.subplot(1,2,2)
plt.plot(epochs_range,train_accs,label="Train Accuracy")
plt.plot(epochs_range,val_accs,label="Validation Accuracy")
plt.title("Accuracy Comparison")
plt.legend()
plt.show()